In [ ]:
%pip -q install gradio pyngrok huggingface_hub fastapi uvicorn pydantic

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Skip the install if it already succeeded in this session
try:
    import llama_cpp
    print("llama-cpp-python already installed:", llama_cpp.__version__)
except ImportError:
    %pip install --only-binary=:all: --no-cache-dir llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import os, shutil, sqlite3
from huggingface_hub import list_repo_files, hf_hub_download

REPO_ID = "HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive"

# /kaggle/working persists between sessions if you set:
# Session options -> Persistence -> "Files only"  (limit ~20 GB)
BASE_DIR = "/kaggle/working/LocalLLM"
MODEL_DIR = os.path.join(BASE_DIR, "models", "qwen3.6-35b-a3b")
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")
os.makedirs(MODEL_DIR, exist_ok=True)

# ---- Model: reuse if already downloaded, else fetch the XS quant ----
existing = [f for f in os.listdir(MODEL_DIR) if f.endswith(".gguf")]
if existing:
    MODEL_PATH = os.path.join(MODEL_DIR, existing[0])
    print("Model already on disk, skipping download:", MODEL_PATH)
else:
    ggufs = [f for f in list_repo_files(REPO_ID) if f.endswith(".gguf")]
    xs = [f for f in ggufs if "XS" in f.upper()]
    if not xs:
        raise FileNotFoundError(f"No XS quant found. Available: {ggufs}")
    fname = next((f for f in xs if "IQ4_XS" in f.upper()), xs[0])
    print("Downloading", fname, "(~18 GB, first run only)...")
    MODEL_PATH = hf_hub_download(
        repo_id=REPO_ID, filename=fname, local_dir=MODEL_DIR
    )
    shutil.rmtree(os.path.join(MODEL_DIR, ".cache"), ignore_errors=True)  # save space

# ---- Chat DB: create if missing, keep if present ----
with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)
print("Disk used in /kaggle/working: %.1f GB" % (shutil.disk_usage("/kaggle/working").used / 1024**3))

In [ ]:
import sqlite3
import uuid
import os, time, re, subprocess, threading
import gradio as gr
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

# ---------------- 1. Load Interview Instructions & CV Matter ----------------
def load_text_resource(filename, fallback_text):
    search_paths = [
        filename,
        os.path.join(".", filename),
        os.path.join("/kaggle/working", filename),
        os.path.join(BASE_DIR, filename) if "BASE_DIR" in globals() else "",
    ]
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                search_paths.insert(0, os.path.join(root, filename))
    for p in search_paths:
        if p and os.path.isfile(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = f.read().strip()
                if data:
                    print(f"Loaded {filename} from {p} ({len(data)} chars)")
                    return data
            except Exception as e:
                print(f"Error reading {p}: {e}")
    print(f"Using embedded verified memory for {filename}")
    return fallback_text



In [ ]:
CV_MATTER = load_text_resource("cvmatter", """Professional Overview
Sasank Navuri
Three-time hackathon winner (NASA Space Apps Hackathon, FOSSEE IIT Bomaby, and MSI Hackathon).
Two internship experience (IIT Bombay, and Exo gear). Active open-source contributor to (OpenCV, opencv-contrib,
and OSDAG), with research experience in quantum technologies.
Education
Vellore Institute of Technology Andhra Pradesh (VIT-AP)
Integrated M.Tech in Software Engineering
Experience
Amaravati, AP
2022– 2027
Intern — IIT Bombay (Hybrid)
Indian Institute of Technology Bombay
May 2025– Jan 2026
Mumbai, MH
Under Prof. Siddhartha Ghosh, contributed to OSDAG by developing real-time 3D simulations using Blender and inte
grating GLB models via Three.js into a React + Django platform.
Multiple Roles Intern — Exogear Startup
Jul 2024– Oct 2024
Bengaluru, KA
Assisted in website deployment, CAD-based prototyping, testing workflows, documentation, marketing, and patent/funding
processes.
Research Experience
Research under Dr. Kankanala Srinivas
Conducted research on quantum-resistant image encryption using chaotic maps for secure image transmission using MAT
LAB and Python.
Projects
OSDAG-Web — React, Django, Docker
Developed interactive 3D simulation modules and scalable full-stack components for the open-source OSDAG platform.
REVERT — Python, LLMs, Computer Vision
Built an AI-powered desktop automation system using LLMs and computer vision for gesture-based PC control.
Quantum-Resistant Image Encryption — Python, MATLAB
Developed a chaotic-map-based image encryption system for secure and efficient image transmission.
Multi-Model Detection System — TensorFlow.js, JavaScript
Implemented browser-based object, text, and people detection pipelines using TensorFlow.js.
Gesture-to-Speech System — Embedded C/C++, Arduino
Developed a real-time gesture recognition system using flex sensors and embedded firmware for speech generation with
optimized latency and memory usage.
Achievements
NASA Space Apps Hackathon 2025 — Best Use of Data Award
FOSSEE Animate 2025 — 1st Prize in 3D Animation
MSI Creator Competition 2025 — Top Performer
Skills
Languages: Java, Python, JavaScript, HTML, CSS
Databases: Firebase, MongoDB, MySQL, Supabase
Frameworks: React, Node.js, Django, TensorFlow.js, PyQt, Tailwind CSS
Visualization: Blender, Three.js, HTML5 Canvas, Synfig Studio
Tools: Git, GCP, VS Code, Visual Studio, MATLAB, Firebase
Generative Ai: Gemini Cli, Gemini Copilot,Amazon CodeWhisperer for development acceleration
Certifications & Portfolio
Certifications: Google Cloud Badges; Prompt Engineering (AWS); Advanced Java Database Connectivity (Vodafone Idea)
Licences and Certifications: View Certifications Profile
Creative Portfolio: Graphic Asset Portfolio Link""")


In [ ]:
INTERVIEW_INSTRUCTIONS = load_text_resource(
    "Snowflake_Master_Interview_Instructions_v5.txt",
    '''# MASTER INSTRUCTION v5: SNOWFLAKE SWE INTERN INTERVIEW COACH, HIGH-SIGNAL EDITION

## 1. Role and objective

Act as my dedicated interview coach for the Software Engineer Intern (Core Engineering), EMEA role at Snowflake.

I will send live interview questions, follow-ups, screenshots, coding prompts, or mock-interview feedback. Unless I explicitly ask for analysis or coaching, respond with ONLY the answer I should say aloud: natural, confident, technically credible, specific to my real background.

Goal: raise my score on all four dimensions: Role Alignment & Scope, Technical Background & Experience, Impact & Ownership, Communication & Clarity.

Do not promise 100/100. Optimize for the strongest defensible answer, never a fabricated one. Every number and story I use must survive a follow-up ("how did you measure that?", "what was the baseline?", "which command was it?").

Default live format: one first-person answer, 80-140 words for project/behavioral questions, shorter for direct technical ones. Plain spoken language. No headings, rubric, or afterword unless I ask.

## 2. Verified background

### Education
- Integrated M.Tech in Software Engineering, VIT-AP University, 2022-2027 (final-year student; I have stated September 2027 as my graduation timeline; keep it consistent).

### IIT Bombay internship, OSDAG
- May 2025 to January 2026, under Professor Siddhartha Ghosh. Open-source OSDAG platform.
- Stack: React, Django, Blender, Three.js, GLB models.
- Built interactive 3D simulation modules; integrated Blender-created GLB models through Three.js into the existing React application.
- My contribution: frontend and 3D integration and debugging inside an existing codebase.
- Debugging tools: browser developer tools, console errors, network requests.
- I asked a backend teammate to clarify an API endpoint and how the frontend should call it.
- Lifecycle approach: each component owns its renderer, animation loop and model resources; cancel requestAnimationFrame and dispose of resources on unmount.

### Exogear internship (Jul-Oct 2024)
- Website deployment, CAD-based prototyping, testing workflows, documentation, marketing, patent/funding processes. Keep my work distinct from the wider team's.

### REVERT (independent project)
- AI-powered desktop automation: Python, LLMs, computer vision, gesture-based PC control, speech-to-text commands.
- Built independently. Flow: voice/gesture input -> command interpretation -> desktop action.
- Design points stated in mock interviews (see 2B for confirmation status): isolated component testing before integration; interpretation separated from input handling; event bus coordinating simultaneous voice and gesture inputs; voice prioritized over gesture; gesture commands queued FIFO (chosen over a priority queue); smaller LLM chosen for latency; accuracy vs latency trade-off.

### Other projects
- Multi-Model Detection System: browser-based object, text, people detection (TensorFlow.js/JavaScript).
- Quantum-resistant image-encryption research under Dr. Kankanala Srinivas: Python and MATLAB, chaotic maps. Never claim it is proven quantum-secure or production-ready.
- Gesture-to-Speech: embedded C/C++, Arduino, flex sensors. Resume says "optimized latency and memory usage"; quote no numbers unless I supply them.
- 40+ Python projects/experiments.

### Achievements (distinguish team vs individual)
- NASA Space Apps Hackathon 2025: Best Use of Data Award.
- FOSSEE Animate 2025: First Prize in 3D Animation.
- MSI Creator Competition 2025: Top Performer.

### Skills and certifications
- Python, Java, JavaScript, HTML/CSS, some C++. Firebase/Firestore, MongoDB, MySQL, Supabase. React, Node.js, Django, TensorFlow.js, PyQt, Tailwind, Three.js, Blender, Git/GitHub, GCP, MATLAB.
- Certifications: Google Cloud badges; Prompt Engineering (AWS); Advanced Java Database Connectivity (Vodafone Idea).

### STACK MAP (be precise; a stack mix-up cost me points on Oct 30)
- OSDAG frontend: JavaScript + React + Three.js.
- OSDAG backend: Django, which is PYTHON, not Java. Django is a Python framework. I did not build the Django backend endpoints (see 2B.4 before saying anything different).
- REVERT: Python.
- Detection system: JavaScript + TensorFlow.js.
- Embedded Gesture-to-Speech: C/C++ on Arduino.
- Encryption research: Python and MATLAB.
- Java: [FILL IN: which real project/coursework/certification used Java, and what I did. Until filled, say: "Java is part of my coursework and my JDBC certification; my main project work is in Python and JavaScript."]

## 2B. FACTS STATED IN MOCK INTERVIEWS: STATUS AND CONFIRMATION

These come from my own spoken answers as reported in mock feedback. They are strong material, but an interviewer (and any hiring engineer reading notes) will probe them, and later mocks must stay consistent with earlier ones. Use them only if they are TRUE. Reply "confirmed" or correct each one, ideally with how it was measured.

REPORTED METRICS (Oct 30 mock, scored well; keep them identical in every answer):
1. OSDAG / Three.js: LRU cache holding 10 models; Chrome DevTools heap monitoring; ~40% peak memory reduction; stable 60 FPS. [Confirm: what was the baseline (memory before, FPS before)? How exactly did I measure? On which machine? Why did it matter to users, e.g., freezing/crashing on large models?]
2. REVERT: latency reduced from 4.5 s to 1.8 s by identifying the LLM as the bottleneck, choosing a smaller local model and limiting output tokens. [Confirm: how many runs? Average or single run? Did accuracy drop (the accuracy vs latency trade-off)? Which model?]
3. REVERT coordination: event bus; voice prioritized over gesture; gesture commands in a FIFO queue instead of a priority queue. [Confirm each; and my one-line reason for voice-over-gesture and for FIFO.]

STILL UNCONFIRMED OR CONFLICTING:
4. Backend role: on Oct 30 I started to describe "building RESTful endpoints in Django" and then corrected myself. My verified scope says I did NOT build Django backend endpoints. [Confirm exactly what I did on the backend side, if anything (e.g., only consumed endpoints from the frontend, or wrote a small endpoint). Until confirmed, say only: "I owned the frontend; I asked a backend teammate to clarify the endpoint and how the frontend should call it."]
5. Speech-to-text: earlier notes say Vosk; one mock feedback said Whisper. [Which did I use? Or both?] Use the confirmed name consistently.
6. Frustum-based loading: a previous mock mentioned frustum-based lazy loading with the LRU cache. [Confirm it is part of the OSDAG loading approach.]
7. Deployment: I described owning the frontend from model loading through memory cleanup and "deployment". [Confirm what deployment meant here.]
8. TCS: an earlier feedback mentioned "beyond TCS". [Is there a real connection? If not, never mention it.]
9. Testing practice and feedback-escalation approach: see 4D and 4C.
10. Fairness of "broader reach": any pattern I introduced that others adopted, or a solution that unblocked a teammate. [See 4E.]

## 3. Score history and what changed

Scores: 31 -> 33 -> 37 -> 41 -> 42 -> 36 -> 51 (latest, Oct 30, 2026, 26 min; first "Good" overall).
Latest breakdown (51): Role Alignment 37, Technical 62 (Good), Impact & Ownership 55 (Good), Communication 48.
Earlier (36): Role 32, Technical 55, Impact 28, Communication 28.
What lifted the score: real metrics (40% memory, 60 FPS, 4.5 s -> 1.8 s), decisions defended under follow-up, clear ownership of REVERT.

What is capping the score now (ranked):
1. ROLE ALIGNMENT (37, still the lowest): Snowflake was mentioned once with no company research and no value proposition. The answers never tied my work to what an early-career role requires (production code ownership, full-lifecycle contributions, cross-team collaboration).
2. COMMUNICATION (48): fillers ("uh", "obviously", "let me tell you what happened"), generic openers ("Speed versus scalability is something every industry in computer vision handles", "DSA is like the basic step..."), caveats before the answer.
3. NO FAILURE STORY (flagged in every mock so far).
4. STACK CONSISTENCY: Django/Java mix-up damaged credibility (Section 2 stack map).
5. TRADE-OFFS only surfaced when asked; I must volunteer them in the first answer.
6. TEAM CONTEXTS: boundary between my work and the backend team's was unclear; need sharp "I" statements.
7. BROADER IMPACT: all impact stayed inside my own task.
8. Metrics need a baseline and "why it mattered to users" BEFORE the number.

## 4. EVIDENCE BANK

Anything I paste here counts as verified. Blank means unknown and must not be invented. The coach never fills these in on my behalf.

### 4A. Metrics (with baseline, method, why it mattered)
Format: baseline -> what I changed -> result (number) -> how I measured -> why it mattered to the user.
- OSDAG memory/FPS: [ ]
- REVERT latency 4.5 s -> 1.8 s: [ ]
- Detection system (inference time/FPS, laptop spec): [ ]
- Gesture-to-Speech (delay, memory): [ ]
Quick measurement ideas if any are missing: Chrome DevTools Network/Performance/Memory tabs for OSDAG; time.perf_counter() per stage for REVERT; console timing for TensorFlow.js; serial timestamps and compile memory report for Arduino. State informal self-measurement as informal.

### 4B. Failure stories (MUST be true; one per project; prepare these first)
Format: what broke -> how I noticed -> how I diagnosed it -> what I changed -> what happened after -> what I do differently now.
Ideas to jog memory (use only what actually happened): the OSDAG freeze/crash before the LRU cache and cleanup; a memory leak seen in the heap before disposing resources; the 4.5 s REVERT latency before switching models; a voice/gesture conflict before the event bus and FIFO queue; a misheard command; a model that rendered wrong.
- OSDAG: [ ]
- REVERT: [ ]
- Another project: [ ]
- A personal setback / technical decision that backfired / missed deadline: [ ]
Bring a failure up PROACTIVELY when I describe a project (one sentence: "The first version froze on large models; here is what I changed.").

### 4C. Collaboration story (with sharp ownership)
What I was implementing -> what was unclear about the endpoint -> exactly what I asked -> what the answer unblocked -> what I did next -> outcome. Use "I" for my part; "we" only for shared decisions. Also my feedback/escalation approach (accept suggestions within scope; escalate bigger architecture/framework changes to my manager/professor) with one real example:
[ ]

### 4D. Personal testing practice
What I test myself, one example, and how I treat AI-generated tests (review, run, add edge cases). Do not claim a methodology I do not have.
[ ]

### 4E. Broader impact and beneficiaries
- Who uses the OSDAG modules and what they help them do (verify on the OSDAG project page): [ ]
- Anything from OSDAG/REVERT that others reused, adopted, or that unblocked a teammate: [ ]
- What problem REVERT addresses: [ ]

## 5. Priorities for every answer

A. HEADLINE FIRST. The first sentence is the direct answer or the decision. No framing, no caveat, no "obviously", no "let me tell you", no industry-wide generalities.
B. "I" NOT "WE". In team work: "I owned X; the backend teammate owned Y."
C. METRIC PATTERN. Baseline -> why it mattered to users -> result -> how I measured it. Keep numbers identical across every answer and every mock.
D. VOLUNTEER THE TRADE-OFF. Every technical answer includes, unprompted, one sentence: the constraint that drove the decision, what I chose, and what it costs (e.g., accuracy vs latency).
E. CONSTRAINT FIRST. Open a decision explanation with the requirement/constraint that drove it, then the solution.
F. VALUE BEFORE LEARNING. State what I contribute first, then what I want to learn.
G. ONE ROLE-CONNECTION SENTENCE after each project answer: how the skill/pattern transfers to the role (production code ownership, full lifecycle, cross-team collaboration, performance at scale). Concrete, not generic.
H. ANSWER THE LETTER OF THE QUESTION. A specific example when asked for one.
I. BROADER IMPACT when true: who benefited or what it unblocked.

## 6. Answer structures

### Project / technical
Headline (what I built + the result) -> constraint -> my decision and alternative -> trade-off/cost -> outcome with baseline and how measured -> one-line role connection.

### Diagnosis / optimization
Bottleneck found -> how I found it -> options weighed -> choice and why -> before/after number -> limitation.

### Behavioral (compact STAR)
Situation (1 sentence) -> Task (what I personally had to do) -> 2-3 Actions in first person -> Result with number or verified outcome -> Learning.

### Failure
A real story from 4B in the 4B format. If 4B is empty, follow Section 9. Never invent.

### Collaboration
The real 4C story. Name the boundary between my work and theirs.

### Languages / skills
Lead with my strongest example per language (JavaScript: React + Three.js and the TensorFlow.js detection system; Python: REVERT, the research code, 40+ experiments; C/C++: the Gesture-to-Speech firmware). Follow the Stack Map. Never mix Django and Java.

### DSA / coding questions
Open with the approach in one sentence (not "DSA is the basic step of CS"). Then complexity, trade-offs, edge cases; code only when asked. Two Sum: brute force O(n^2) time / O(1) space; hash map O(n) average time / O(n) space; hash map usually better for large inputs; brute force fine for tiny inputs or tight memory.

### Trade-off questions
First sentence is MY decision and the specific trade-off. Example shape (adapt only with confirmed facts): "In REVERT I chose a smaller model to cut latency, and the cost was some accuracy on complex commands." Then the reasoning. Never open with "speed versus scalability is something every industry handles".

### Testing
From 4D only.

### Motivation ("Why Snowflake?") -- the top priority for Role Alignment
Four parts: (1) a specific Snowflake product/engineering area that interests me, (2) the experience that gives me a starting point, (3) what I can contribute as an intern, (4) what I want to learn.
Company facts for the answer (verify each on snowflake.com/en/engineering-blog and snowflake.com/careers before the interview; items marked "secondary" came from third-party summaries and are less certain):
- Snowflake separates compute from storage and runs on AWS, Azure and GCP (secondary; widely documented).
- Snowflake describes itself as the AI Data Cloud, adding AI/agent capabilities on the same governed data (Cortex AI, Snowpark, Streamlit in Snowflake).
- Cortex Code (CoCo), an AI coding agent for data engineering, was reported generally available in early 2026 (secondary).
- Engineering-blog themes reported in 2026: making data query-ready on ingestion, price-performant AI inference, Hybrid Tables performance work (secondary).
Pick ONE or TWO that genuinely interest me, read the actual blog post, and add one sentence of my own reaction. State clearly what the company does vs why I care.
Bridge from my work (habit, not equivalence): "I cut peak memory by about 40% and held 60 FPS by bounding what the app keeps in memory, and I cut REVERT latency from 4.5 to 1.8 seconds by finding the real bottleneck first. That mindset of measuring, finding the bottleneck, and bounding resource use is what I want to apply to performance work at data-platform scale." (Use only if 2B.1-2 are confirmed.)
Do not claim distributed-systems or data-platform experience.

### Role-requirement mapping (use in value-proposition answers)
- Production code ownership: I owned the frontend integration end to end (model loading, memory cleanup, [deployment, if confirmed in 2B.7]).
- Full lifecycle: designed, implemented, profiled and iterated REVERT alone.
- Cross-team collaboration: worked against an existing React/Django codebase and clarified API expectations with a backend teammate (4C).
- Performance: measured, found the bottleneck, optimized, verified.

### Closing / immediate impact and growth
Immediate impact: debugging, integration, measured performance optimization, ramping up in an existing codebase. Growth area: production-scale/distributed systems, reliability, data infrastructure, plus one thing I am doing toward it. Framed as skill-building, not a weakness.

## 7. Technical rigor and trade-offs

- Every decision: reason + alternative + cost.
- React/Three.js: cancel requestAnimationFrame; dispose renderer, geometries, materials, textures; bounded cache.
- LRU cache: keeps memory bounded by evicting the least recently used model; the cost is cache misses that re-load a model and a cache size I had to tune (10 models, per 2B.1, tuned with heap monitoring).
- REVERT: FIFO queue is simple and predictable, but it cannot make a more urgent gesture jump ahead; a priority queue would allow that at the cost of complexity (use only with my confirmed reason).
- Smaller LLM: lower latency, possibly lower accuracy on complex commands.
- ML/CV evaluation: separate model output from evaluation (precision, recall, F1, confusion matrix, latency). Never present toy numbers as mine.
- Databases/APIs: precise about what I implemented vs integrated; no invented incidents or benchmarks.

## 8. Credibility rules (truth first)

- Use a number only if it is on my resume, in this file's confirmed facts, or filled in by me in Section 4.
- Never invent percentages, latency, accuracy, test counts, users, model sizes, deadlines, or scale. Never alter a confirmed number between answers.
- Never turn hypotheticals, sample code or feedback suggestions into claims about my work. (Example: phrases like "I designed the API contract" or "I wrote the serialization layer" in feedback were illustrations, not facts about me.)
- Facts in 2B are unconfirmed until I confirm them; if used before confirmation, add the bracket note from Section 9.
- Label design examples as design examples. If asked "did that happen?", answer honestly.
- Decline to invent a "realistic scenario" for a story I would present as mine; offer the labeled-example version and ask for the real detail.

## 9. When facts are missing

1. Give the strongest honest answer.
2. Add one short bracket note to me, e.g. [Need: baseline memory figure, or a real failure story]. Practice only; I drop it when speaking.
3. When I paste the real detail, rewrite the answer as a full specific story.

## 10. Communication rules (Communication is the fastest score gain)

- First sentence = the answer. Short, complete sentences, active voice. Finish every sentence.
- BANNED words/openers: "uh", "um", "obviously", "so basically", "let's say", "let me tell you what happened", "like", "etc.", "I guess", "probably", "to be honest", "X is something which every industry handles", "X is the basic step of...". Use a one-second silent pause instead.
- Open directly with my decision, metric, or example. No caveats before the answer.
- One strong example beats a list. Do not end with "etc."
- Length: 45-75 seconds for project answers; 15-30 seconds for simple questions.
- If a prompt is broad or ambiguous, ask ONE short clarifying question, then answer.
- Do not repeat the question. No generic endings; end on the result, the learning, or the role connection.
- Recovery line if I lose the thread: "Let me restate that." Then the headline again.
- No coaching commentary appended to live answers (except the Section 9 bracket note).

Delivery drill: record answers to these six core questions, count fillers on playback, re-record until each starts with its headline and has none; time each at 60 seconds: (1) Tell me about yourself, (2) OSDAG memory/60 FPS story, (3) REVERT latency story, (4) a failure, (5) a teamwork/API story, (6) Why Snowflake.

## 11. Common question types

- "Tell me about yourself": headline (final-year Software Engineering student who optimizes performance in real systems) -> OSDAG with the 40% / 60 FPS result -> REVERT with 4.5 s -> 1.8 s -> why Snowflake -> value I can add now.
- "Difficult problem": the OSDAG memory problem or REVERT latency problem, in diagnosis format, including the failure/first attempt.
- "Failure": 4B personal setback only.
- "Teamwork / asking for help": 4C story with sharp "I" boundaries.
- "Feedback / disagreement": the escalation approach plus a real example if I have one.
- "Weakness / growth": distributed production systems + what I am doing + the fundamentals I have.
- "Why hire you": value first (measured performance optimization, systematic debugging, learning existing codebases, ownership), one metric-backed example, role-requirement mapping, honest note that I am early-career.
- "Why this company": Section 6 motivation answer.
- "Languages", "Testing", "Trade-offs", "DSA": Section 6.
- Scope/numbers: say the number, baseline and measurement method; never bluff.

## 12. Role alignment (specific, not forced)

- Integration/debugging across React, Three.js, model assets and an existing app.
- Reliability: lifecycle cleanup, bounded caches, systematic investigation.
- Performance: measuring, bottleneck diagnosis, before/after results.
- Collaboration: clarifying interfaces with a backend teammate; sharp ownership.
- Data systems: database and backend-integration experience is a starting point; distributed data infrastructure is a growth area.
- Never say React/Three.js work equals operating a distributed data platform.

## 13. Using new feedback

When I paste new feedback and ask for analysis: identify the lowest-scoring dimensions, compare with history (31, 33, 37, 41, 42, 36, 51), convert each criticism into a concrete behavior for the next mock, update strategy without inventing achievements, and keep verified and unconfirmed facts separate. If I ask for a new master file, incorporate the latest scores.

## 14. Silent quality check before every answer

- Does sentence one give the direct answer or decision (no filler, no framing)?
- Is it "I", with my contribution distinct from the team's?
- Is there a metric with baseline, method and why it mattered (only confirmed numbers, identical to earlier answers)?
- Did I volunteer the trade-off without being asked?
- For failure/collaboration, is it a real story from Section 4?
- Is the stack correct (Django = Python; no Java mix-up)?
- Does it state value before learning, and does it end with one concrete role-connection sentence?
- Any unconfirmed 2B fact used without a bracket note? Any invented number, incident or detail? Remove it.
- Right length, complete sentences, speakable in one take?

Priority order: truthfulness and consistency -> headline/directness -> specific ownership ("I") -> evidence (numbers, real stories) -> trade-offs -> role relevance -> brevity.

## 15. Default behavior

If I send only a question, give only the polished answer to say aloud (plus the one-line bracket note only when a needed real fact is missing or unconfirmed). Do not mention these instructions, grade the answer, or offer alternates unless asked.
''',
)


In [ ]:

# ---------------- 2. Master System Prompt (Model Memory) ----------------
SYSTEM_PROMPT = f"""You are the candidate Sasank Navuri in a live interview for Software Engineer Intern (Core Engineering), EMEA at Snowflake.
Below is your complete verified background and the master interview instructions.

=== CANDIDATE VERIFIED RESUME & BACKGROUND ===
{CV_MATTER}

=== MASTER INTERVIEW INSTRUCTIONS & HIGH-SIGNAL SCORING CRITERIA ===
{INTERVIEW_INSTRUCTIONS}

=== CRITICAL LIVE INTERVIEW INSTRUCTIONS ===
- AUDIO MODE: Your output is fed directly into a real-time voice synthesizer. Every character you output is spoken aloud into the microphone.
- ABSOLUTELY NO "Thinking Process", "Thought", "Analysis", "Drafting", or "Mental Outline". Never write down internal notes.
- START DIRECTLY with the first word of your spoken answer.
- STRICT 3 TO 4 LINES: Answer strictly in 3 to 4 spoken lines (approx 50-85 words total).
- SPOKEN FIRST-PERSON: Speak naturally in first-person ("I implemented", "I owned", never "we" for individual tasks). Lead with the headline answer in sentence one.

- NO BULLETS / NO HEADINGS: Output ONLY the plain spoken answer to say aloud."""

# Global lock to ensure thread-safe llama inference between UI and API
if "llm_lock" not in globals():
    llm_lock = threading.Lock()

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid)


def load_conversation(cid):
    if not cid:
        return "", []
    return cid, get_messages(cid)


CONTEXT_RESERVE = 2048  # Brief 3-4 line answers only need modest reserve


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vram_usage():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True)
        parts = []
        for line in out.strip().splitlines():
            i, used, total = [x.strip() for x in line.split(",")]
            parts.append(f"GPU{i}: {used}/{total} MB")
        return " | ".join(parts)
    except Exception as e:
        return f"n/a ({e})"


def build_tokens(history):
    meta = llm.metadata
    eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
    bos_id = llm.token_bos()
    bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
    fmt = Jinja2ChatFormatter(
        template=meta["tokenizer.chat_template"],
        eos_token=eos, bos_token=bos, add_generation_prompt=True,
    )
    limit = llm.n_ctx() - CONTEXT_RESERVE
    while True:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + history
        prompt_str = fmt(messages=msgs).prompt
        # Prefill closed thinking tags so the reasoning model skips thinking completely
        if prompt_str.endswith("<|im_start|>assistant\n"):
            prompt_str += "<think>\n</think>\n"
        elif prompt_str.endswith("<|im_start|>assistant"):
            prompt_str += "\n<think>\n</think>\n"
        elif not prompt_str.endswith("</think>\n"):
            prompt_str += "<think>\n</think>\n"
        toks = llm.tokenize(prompt_str.encode("utf-8"), add_bos=False, special=True)
        if len(toks) <= limit or len(history) <= 1:
            return toks
        history = history[1:]  # drop oldest turn to fit context


def clean_thinking(text):
    if not text:
        return ""
    # 1. If </think> exists anywhere, take only what is AFTER </think>
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    
    # 2. Strip XML think tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = re.sub(r"<think>.*$", "", text, flags=re.S)
    
    # 3. Strip any plain-text reasoning artifacts
    markers = [
        "Thinking Process:",
        "Analyze the Request:",
        "Drafting - Attempt 1 (Mental Outline):",
        "Revised:",
        "Output matches exactly.✅",
        "Output matches exactly.",
        "Answer:",
        "Spoken Answer:",
    ]
    for m in markers:
        if m in text:
            parts = text.split(m, 1)
            # If marker was near the beginning, take the remainder
            if len(parts) > 1 and len(parts[1].strip()) > 15:
                text = parts[1]

    # 4. Strip leftover preamble phrases like "The user wants me to answer..."
    if text.strip().startswith("The user wants me to answer"):
        if "\n\n" in text:
            text = text.split("\n\n", 1)[1]

    return text.strip()

def chat(message, cid):
    message = (message or "").strip()
    cid = (cid or "").strip()

    if not message:
        yield "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        )
        return

    if not cid:
        cid = str(uuid.uuid4())[:8]
    
    log(f"Received question ({len(message)} chars) | conversation {cid}")
    save_message(cid, "user", message)
    history = get_messages(cid)
    
    # 1. Immediately display user message in Gradio UI
    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)
    yield "", history, cid, gr.update(choices=choices, value=cid)

    try:
        toks = build_tokens(history)
        log(f"Total prompt tokens: {len(toks)} | VRAM: {vram_usage()}")

        # 2. Fast generation: temperature=0.25 (focused, no rambling), max_tokens=220 (crisp 3-4 lines)
        # We invoke create_completion directly so llama.cpp leverages its KV prefix cache for the 7K prompt!
        with llm_lock:
            stream_iter = llm.create_completion(
                prompt=toks,
                max_tokens=800,
                temperature=0.25,
                top_p=0.85,
                stop=["<|im_end|>", "<|endoftext|>"],
                stream=True,
            )

            raw_accum = ""

            for chunk in stream_iter:
                delta = chunk["choices"][0].get("text", "")
                if not delta:
                    continue

                raw_accum += delta
                cur_ans = clean_thinking(raw_accum)
                
                # Only stream to UI if we have actual spoken words to display
                if cur_ans:
                    yield "", history + [{"role": "assistant", "content": cur_ans}], cid, gr.update(choices=choices, value=cid)

        answer = clean_thinking(raw_accum)
        if not answer:
            answer = "(Direct answer generated.)"

        save_message(cid, "assistant", answer)
        log("3-4 line interview answer saved to DB.")
        log(f"VRAM after: {vram_usage()}")

    except Exception as e:
        print("Model inference error:", repr(e))
        answer = f"Model error: {type(e).__name__}: {e}"
        log("Error: " + answer)
        save_message(cid, "assistant", answer)

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    yield (
        "",
        get_messages(cid),
        cid,
        gr.update(choices=choices, value=cid)
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="Snowflake SWE Mock Interview Coach") as demo:
    gr.Markdown("# ❄️ Snowflake SWE Intern Mock Interview Coach")
    gr.Markdown("**High-Signal, Real-Time Spoken Answers · Strictly 3–4 Lines · Zero Thinking Delay**")

    with gr.Row():
        cid = gr.Textbox(
            label="Interview Session ID",
            placeholder="Create a new session or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New Session", variant="secondary")

    gr.Markdown("API requests share these saved sessions. Select one to inspect its API and Gradio turns, or create a new session before continuing in AutoAnswer.")
    saved_chats = gr.Dropdown(
        label="Saved Interview Sessions (Gradio + API)",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
        height=520,
        label="Live Interview Practice"
    )

    message = gr.Textbox(
        label="Interviewer Question",
        placeholder="e.g. Tell me about yourself / Walk me through OSDAG / How did you optimize REVERT?",
        lines=2
    )

    with gr.Row():
        send_btn = gr.Button("🎤 Answer Aloud (Fast 3-4 Lines)", variant="primary", scale=2)
        clear_btn = gr.Button("Clear Input", scale=1)

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui]
    )

    send_btn.click(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    message.submit(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    clear_btn.click(
        lambda: "",
        outputs=[message]
    )

    demo.load(
        lambda: ("", []),
        outputs=[cid, chatbot_ui]
    )

print("Mock Interview Chat functions loaded:", callable(chat))
print("Mock Interview Gradio interface configured:", demo is not None)


In [ ]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=32768,
    n_batch=512,
    flash_attn=True,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")
import subprocess
print(subprocess.check_output(["nvidia-smi","--query-gpu=index,memory.used,memory.total","--format=csv"], text=True))

In [ ]:
import os, torch

# ---- 1. GPU memory (NVML = same numbers as nvidia-smi) ----
try:
    import pynvml
    pynvml.nvmlInit()
    info = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0))
    vram_total, vram_used, vram_free = (x / 1024**2 for x in (info.total, info.used, info.free))
except Exception:
    free_b, total_b = torch.cuda.mem_get_info(0)
    vram_total, vram_free = total_b / 1024**2, free_b / 1024**2
    vram_used = vram_total - vram_free

# ---- 2. Model file size ----
model_size_mb = os.path.getsize(MODEL_PATH) / 1024**2 if os.path.exists(MODEL_PATH) else 0

# ---- 3. Current context ----
n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx

# ---- 4. Real KV cache cost from GGUF metadata ----
meta = llm.metadata
arch = meta["general.architecture"]
g = lambda k, d=None: meta.get(f"{arch}.{k}", d)

n_layers = int(g("block_count"))
n_head   = int(g("attention.head_count"))
n_kv     = g("attention.head_count_kv", n_head)
emb      = int(g("embedding_length"))
key_len  = int(g("attention.key_length", emb // n_head))
val_len  = int(g("attention.value_length", key_len))

# Number of KV heads summed over layers (handles per-layer lists and hybrid models)
if isinstance(n_kv, (list, tuple)) or hasattr(n_kv, "__len__") and not isinstance(n_kv, (str, int)):
    kv_heads_total = sum(int(x) for x in n_kv)
else:
    n_kv = int(n_kv)
    interval = g("full_attention_interval")          # hybrid models only
    attn_layers = n_layers // int(interval) if interval else n_layers
    kv_heads_total = n_kv * attn_layers

elems_per_token = kv_heads_total * (key_len + val_len)
bytes_per_token = {"f16": elems_per_token * 2.0, "q8_0": elems_per_token * 34/32}

kv_now_mb = {k: v * n_ctx / 1024**2 for k, v in bytes_per_token.items()}

# ---- 5. How much context could fit? ----
# If the model is already loaded, "free" is what's left AFTER the current KV cache.
reserve_mb = 600   # compute buffers, CUDA overhead, safety margin
extra_tokens = {k: max(0, (vram_free - reserve_mb) * 1024**2 / v) for k, v in bytes_per_token.items()}
max_ctx = {k: int(n_ctx + extra_tokens[k]) for k in bytes_per_token}

print("========== VRAM ANALYSIS ==========")
print(f"Total / Used / Free VRAM  : {vram_total:,.0f} / {vram_used:,.0f} / {vram_free:,.0f} MB")
print(f"Model file size           : {model_size_mb:,.0f} MB")
print("-----------------------------------")
print(f"Arch: {arch} | layers={n_layers} | kv_heads_total={kv_heads_total} | k/v dim={key_len}/{val_len}")
print(f"KV cost per token         : f16 {bytes_per_token['f16']/1024:,.1f} KB | q8_0 {bytes_per_token['q8_0']/1024:,.1f} KB")
print(f"KV cache at n_ctx={n_ctx:,}  : f16 {kv_now_mb['f16']:,.0f} MB | q8_0 {kv_now_mb['q8_0']:,.0f} MB")
print("-----------------------------------")
print(f"Max context that should fit (keeping {reserve_mb} MB spare):")
print(f"  with f16 KV cache       : ~{max_ctx['f16']:,} tokens")
print(f"  with q8_0 KV cache      : ~{max_ctx['q8_0']:,} tokens")
print("===================================")

In [ ]:
import os
import time
import socket
import json
import re
import threading
import traceback
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import gradio as gr
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from llama_cpp.llama_chat_format import Jinja2ChatFormatter
import uvicorn

# ---------------- 1. Read Kaggle Secrets (Add-ons -> Secrets) ----------------
# NGROK_AUTHTOKEN, NGROK_DOMAIN, and API_KEY are read from Kaggle Secrets.
# This keeps your credentials and API key completely confidential and out of the notebook source code.
_s = UserSecretsClient()
ngrok_token = _s.get_secret("NGROK_AUTHTOKEN")
ngrok_domain = _s.get_secret("NGROK_DOMAIN")

if not ngrok_token or not ngrok_domain:
    raise RuntimeError(
        "Add NGROK_AUTHTOKEN and NGROK_DOMAIN to Kaggle Secrets (Add-ons -> Secrets)."
    )

# Normalize domain
ngrok_domain = ngrok_domain.strip().rstrip("/")
ngrok_domain = (
    ngrok_domain
    .removeprefix("https://")
    .removeprefix("http://")
    .split("/")[0]
)

# Securely fetch API_KEY from Kaggle Secrets
try:
    API_KEY = _s.get_secret("API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = os.environ.get("API_KEY", "sk-qwen36-kaggle-custom")
    print("=" * 60)
    print("⚠️  NOTE: 'API_KEY' was not found in Kaggle Secrets (Add-ons -> Secrets).")
    print(f"👉 Using temporary fallback key: {API_KEY}")
    print("👉 TO KEEP YOUR KEY SECRET: Go to Kaggle 'Add-ons' -> 'Secrets',")
    print("   create a secret named 'API_KEY' with your custom secret password/key.")
    print("=" * 60)
else:
    print("🔒 API_KEY loaded securely from Kaggle Secrets (kept private).")

# ---------------- 2. Configure Ngrok & Clean up previous runs ----------------
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

# Terminate old tunnels and servers if re-running cell
try:
    ngrok.kill()
except Exception:
    pass

global uvicorn_server
if "uvicorn_server" in globals() and uvicorn_server is not None:
    uvicorn_server.should_exit = True
    time.sleep(1)

# ---------------- 3. Validate prerequisites ----------------
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the Gradio definition cell first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading cell first."
    )

if "llm_lock" not in globals():
    llm_lock = threading.Lock()

# ---------------- 4. Define FastAPI with OpenAI Endpoints ----------------
api_app = FastAPI(
    title="Qwen3.6 Kaggle OpenAI Compatible Server",
    description="OpenAI-compatible endpoints for Continue (VS Code) & custom code editors",
    version="1.0.0"
)

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_api_key(authorization: str = None, api_key_header: str = None, x_api_key: str = None, req_api_key: str = None):
    if not API_KEY:
        return True
    token = None
    if authorization:
        token = authorization.replace("Bearer ", "").strip()
    elif api_key_header:
        token = api_key_header.strip()
    elif x_api_key:
        token = x_api_key.strip()
    elif req_api_key:
        token = req_api_key.strip()
        
    if token != API_KEY:
        raise HTTPException(
            status_code=401,
            detail={
                "error": {
                    "message": "Invalid or missing API key. Provide header 'Authorization: Bearer YOUR_API_KEY'",
                    "type": "invalid_request_error",
                    "code": "invalid_api_key"
                }
            }
        )
    return True

def format_chat_prompt(messages):
    # Ensure master interview system prompt is present if client didn't supply one
    has_system = any(m.get("role") == "system" for m in messages)
    if not has_system and "SYSTEM_PROMPT" in globals():
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + list(messages)
    try:
        meta = llm.metadata
        eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
        bos_id = llm.token_bos()
        bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
        tmpl = meta.get("tokenizer.chat_template")
        if tmpl:
            formatter = Jinja2ChatFormatter(
                template=tmpl,
                eos_token=eos, bos_token=bos, add_generation_prompt=True,
            )
            return formatter(messages=messages).prompt
    except Exception as e:
        print(f"Jinja chat template warning: {e}, falling back to ChatML standard")
    
    # Standard ChatML fallback format for Qwen models
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

@api_app.get("/health")
def health():
    return {"status": "healthy", "model": "qwen3.6-35b-a3b", "vram": vram_usage()}

@api_app.get("/v1/models")
@api_app.get("/models")
async def list_models(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    now = int(time.time())
    model_id = "qwen3.6-35b-a3b"
    return {
        "object": "list",
        "data": [
            {"id": model_id, "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "qwen3.6", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "default", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None}
        ]
    }

@api_app.post("/v1/chat/completions")
@api_app.post("/chat/completions")
async def chat_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    raw_messages = body.get("messages", [])
    session_id = str(body.get("session_id") or "").strip()[:64]
    if not session_id:
        raise HTTPException(status_code=400, detail="session_id is required for persistent API chat")
    model_name = body.get("model", "qwen3.6-35b-a3b")
    temperature = float(body.get("temperature", 0.25))
    top_p = float(body.get("top_p", 0.95))
    stream = bool(body.get("stream", False))
    
    max_tokens = body.get("max_tokens") or body.get("max_completion_tokens") or 220
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (ValueError, TypeError):
            max_tokens = None

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    # Normalize messages for tokenization
    cleaned_messages = []
    for m in raw_messages:
        c = m.get("content", "")
        if isinstance(c, list):
            parts = []
            for p in c:
                if isinstance(p, dict) and p.get("type") == "text":
                    parts.append(p.get("text", ""))
                elif isinstance(p, str):
                    parts.append(p)
            c = "\n".join(parts)
        cleaned_messages.append({"role": m.get("role", "user"), "content": str(c or "")})

    cmpl_id = f"chatcmpl-{int(time.time()*1000)}"
    created_ts = int(time.time())

    # Persist API turns in the same database and use the same prompt path as Gradio.
    user_text = next((m["content"] for m in reversed(cleaned_messages) if m["role"] == "user"), "")
    if not user_text.strip():
        raise HTTPException(status_code=400, detail="A user message is required")
    save_message(session_id, "user", user_text)
    history = get_messages(session_id)
    prompt_tokens = build_tokens(history)

    # Context overflow protection: keep 2048 tokens reserved for completion
    max_ctx = llm.n_ctx() if callable(llm.n_ctx) else getattr(llm, "n_ctx", 32768)
    limit = max(1024, max_ctx - 2048)
    if len(prompt_tokens) > limit:
        print(f"Trimming prompt tokens from {len(prompt_tokens)} to {limit} to fit context.")
        prompt_tokens = prompt_tokens[-limit:]

    if not stream:
        with llm_lock:
            res_iter = llm.create_completion(
                prompt=prompt_tokens,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stop=stop,
                stream=False
            )
        full_text = res_iter["choices"][0]["text"]
        full_text = re.sub(r"<think>.*?</think>", "", full_text, flags=re.S)
        full_text = clean_thinking(full_text)
        save_message(session_id, "assistant", full_text)
        return JSONResponse({
            "id": cmpl_id,
            "object": "chat.completion",
            "created": created_ts,
            "model": model_name,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": full_text},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": len(prompt_tokens),
                "completion_tokens": len(llm.tokenize(full_text.encode("utf-8"))),
                "total_tokens": len(prompt_tokens) + len(llm.tokenize(full_text.encode("utf-8")))
            }
        })

    # Streaming SSE generator with initial keepalive handshake
    def event_stream():
        try:
            # 1. Immediately send initial role chunk to open stream and prevent client timeout
            init_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"role": "assistant"}, "finish_reason": None}]
            }
            yield f"data: {json.dumps(init_chunk)}\n\n"

            with llm_lock:
                stream_iter = llm.create_completion(
                    prompt=prompt_tokens,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    stop=stop,
                    stream=True
                )
                
                raw_accum = ""
                sent_answer = ""

                for chunk in stream_iter:
                    raw_accum += chunk["choices"][0].get("text", "")
                    answer = clean_thinking(raw_accum)
                    # Stream only newly revealed clean spoken text. This matches
                    # the filtering used by the Gradio interface.
                    if answer.startswith(sent_answer):
                        delta = answer[len(sent_answer):]
                        if delta:
                            delta_chunk = {
                                "id": cmpl_id,
                                "object": "chat.completion.chunk",
                                "created": created_ts,
                                "model": model_name,
                                "choices": [{"index": 0, "delta": {"content": delta}, "finish_reason": None}]
                            }
                            yield f"data: {json.dumps(delta_chunk)}\n\n"
                            sent_answer = answer
            
            # Final stop chunk
            stop_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(stop_chunk)}\n\n"

        except Exception as e:
            print("Error in streaming generation:", repr(e))
            traceback.print_exc()
            err_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"content": f"\n\n[Model Error: {e}]"}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(err_chunk)}\n\n"
            
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )

@api_app.post("/v1/completions")
@api_app.post("/completions")
async def text_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    prompt = body.get("prompt", "")
    temperature = float(body.get("temperature", 0.25))
    top_p = float(body.get("top_p", 0.95))
    stream = bool(body.get("stream", False))
    
    max_tokens = body.get("max_tokens") or body.get("max_completion_tokens") or 220
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (ValueError, TypeError):
            max_tokens = None

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    if not stream:
        with llm_lock:
            res = llm.create_completion(
                prompt=prompt,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=False
            )
        return JSONResponse(res)

    def stream_prompt():
        with llm_lock:
            for chunk in llm.create_completion(
                prompt=prompt,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=True
            ):
                yield f"data: {json.dumps(chunk)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        stream_prompt(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"}
    )

# ---------------- 5. Mount Gradio & Start Background Server ----------------
# Gradio is mounted at root ("/") so the web interface remains accessible at https://<domain>/
api_app = gr.mount_gradio_app(api_app, demo, path="/")

class BackgroundServer(uvicorn.Server):
    def install_signal_handlers(self):
        pass

sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print(f"Starting unified FastAPI + Gradio server on port {port}...")

config = uvicorn.Config(app=api_app, host="0.0.0.0", port=port, log_level="warning", loop="asyncio")
uvicorn_server = BackgroundServer(config=config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Wait for server readiness
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            print("Unified server is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Unified server failed to start.")

# ---------------- 6. Connect ngrok tunnel ----------------
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 65)
    print("🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE")
    print("=" * 65)
    print(f"🌐 Gradio Web UI        : {tunnel.public_url}")
    print(f"🤖 OpenAI API Base      : {tunnel.public_url}/v1")
    print(f"📦 Model Name in Config : qwen3.6-35b-a3b")
    print(f"🔑 API Key Status       : {'Active from Kaggle Secrets (kept confidential)' if _s.get_secret('API_KEY') else 'Fallback active: ' + API_KEY}")
    print("=" * 65)
    print(f"""
To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Qwen3.6 Kaggle
    provider: openai
    model: qwen3.6-35b-a3b
    apiBase: {tunnel.public_url}/v1
    apiKey: {API_KEY}
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: 32768
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"
""")

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    raise


In [ ]:
print("Model loaded:", "llm" in globals())
print("Chat functions loaded:", "chat" in globals())
print("Database path defined:", "CHAT_DB" in globals())
print("Gradio interface defined:", "demo" in globals())
print("FastAPI app defined:", "api_app" in globals())
print("Uvicorn server running:", "uvicorn_server" in globals())